# Fine-tuning SLM Legal Assistant Indonesia

Notebook ini memenuhi kriteria fine-tuning: dataset instruction Alpaca Indonesia, mapping Hugging Face `datasets` ke chat template Unsloth, QLoRA 4-bit double quantization, LoRA pada komponen attention dan FFN, `SFTTrainer` minimal 800 steps, dua eksperimen hyperparameter, evaluasi berkala, dan push model merged 16-bit ke Hugging Face.


In [9]:
# Jalankan di Colab/Kaggle GPU. Restart runtime setelah instalasi bila diminta.
!pip install -q "unsloth[colab-new]" "trl>=0.9.6" "transformers>=4.43.0" "datasets>=2.20.0" "accelerate>=0.33.0" "bitsandbytes>=0.43.1" "huggingface_hub>=0.24.0" "python-dotenv>=1.0.1" "sentencepiece" "protobuf"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

In [ ]:
import os
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
HF_USERNAME = os.getenv("HF_USERNAME")
BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

if HF_TOKEN:
    login(token=HF_TOKEN)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [11]:
raw_dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")
print(raw_dataset)
print("Kolom dataset:", raw_dataset.column_names)

def has_question_and_answer(row):
    question = (row.get("instruction") or row.get("input") or "").strip()
    answer = (row.get("output") or "").strip()
    return bool(question and answer)

raw_dataset = raw_dataset.filter(has_question_and_answer)
raw_dataset = raw_dataset.train_test_split(test_size=0.03, seed=42)

train_raw = raw_dataset["train"]
valid_raw = raw_dataset["test"]

print(train_raw)
print(valid_raw)
print(train_raw[0])

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

alpaca-gpt4-indonesia.csv: reconstructing file:   0%|          |  0.00B / 41.4MB            

alpaca-gpt4-indonesia.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Dataset({
    features: ['Unnamed: 0', 'input', 'output'],
    num_rows: 49969
})
Kolom dataset: ['Unnamed: 0', 'input', 'output']


Filter:   0%|          | 0/49969 [00:00<?, ? examples/s]

Dataset({
    features: ['Unnamed: 0', 'input', 'output'],
    num_rows: 48469
})
Dataset({
    features: ['Unnamed: 0', 'input', 'output'],
    num_rows: 1500
})
{'Unnamed: 0': 2525, 'input': 'Tulis ulang kalimat berikut untuk membuat artinya lebih tepat.\nGadis itu melompat di atas tempat tidur.', 'output': 'Gadis muda melompat naik dan turun di atas kasur.'}


In [12]:
def alpaca_to_messages(example):
    user_content = (example.get("instruction") or example.get("input") or "").strip()

    if example.get("instruction") and example.get("input") and example["input"].strip():
        user_content += "\n\nKonteks tambahan:\n" + example["input"].strip()

    return {
        "messages": [
            {
                "role": "system",
                "content": "Anda adalah asisten AI legal internal berbahasa Indonesia yang menjawab akurat, ringkas, dan bertanggung jawab."
            },
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": example["output"].strip()},
        ]
    }

train_messages = train_raw.map(alpaca_to_messages, remove_columns=train_raw.column_names)
valid_messages = valid_raw.map(alpaca_to_messages, remove_columns=valid_raw.column_names)

print(train_messages[0])

Map:   0%|          | 0/48469 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

{'messages': [{'role': 'system', 'content': 'Anda adalah asisten AI legal internal berbahasa Indonesia yang menjawab akurat, ringkas, dan bertanggung jawab.'}, {'role': 'user', 'content': 'Tulis ulang kalimat berikut untuk membuat artinya lebih tepat.\nGadis itu melompat di atas tempat tidur.'}, {'role': 'assistant', 'content': 'Gadis muda melompat naik dan turun di atas kasur.'}]}


In [13]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

quant_config = getattr(model.config, "quantization_config", None)
print("Quantization config:", quant_config)

if quant_config is not None:
    if isinstance(quant_config, dict):
        print("load_in_4bit:", quant_config.get("load_in_4bit"))
        print("bnb_4bit_use_double_quant:", quant_config.get("bnb_4bit_use_double_quant"))
        print("bnb_4bit_quant_type:", quant_config.get("bnb_4bit_quant_type"))
    else:
        print("load_in_4bit:", getattr(quant_config, "load_in_4bit", None))
        print("bnb_4bit_use_double_quant:", getattr(quant_config, "bnb_4bit_use_double_quant", None))
        print("bnb_4bit_quant_type:", getattr(quant_config, "bnb_4bit_quant_type", None))

def apply_chat_template(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_dataset = train_messages.map(apply_chat_template)
valid_dataset = valid_messages.map(apply_chat_template)

print(train_dataset[0]["text"][:1500])

==((====))==  Unsloth 2026.7.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Quantization config: {'bnb_4bit_compute_dtype': torch.float16, 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': None, 'llm_int8_threshold': 6.0, 'load_in_4bit': True, 'load_in_8bit': False, 'quant_method': 'bitsandbytes'}
load_in_4bit: True
bnb_4bit_use_double_quant: True
bnb_4bit_quant_type: nf4


Map:   0%|          | 0/48469 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

<|im_start|>system
Anda adalah asisten AI legal internal berbahasa Indonesia yang menjawab akurat, ringkas, dan bertanggung jawab.<|im_end|>
<|im_start|>user
Tulis ulang kalimat berikut untuk membuat artinya lebih tepat.
Gadis itu melompat di atas tempat tidur.<|im_end|>
<|im_start|>assistant
Gadis muda melompat naik dan turun di atas kasur.<|im_end|>



In [14]:
# QLoRA adapter pada Multi-Head Attention dan Feed Forward Network.
def build_lora_model(model, r=16, alpha=16, dropout=0.05):
    return FastLanguageModel.get_peft_model(
        model,
        r=r,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=alpha,
        lora_dropout=dropout,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
        use_rslora=False,
        loftq_config=None,
    )

model = build_lora_model(model, r=16, alpha=16, dropout=0.05)


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.5 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [15]:
experiments = [
    {"name": "exp1_r16_lr2e-4", "learning_rate": 2e-4, "r": 16, "alpha": 16, "dropout": 0.05},
    {"name": "exp2_r32_lr1e-4", "learning_rate": 1e-4, "r": 32, "alpha": 32, "dropout": 0.05},
]

# Untuk percobaan cepat perbandingan loss, jalankan 120 step dahulu. Untuk submission final, FINAL_MAX_STEPS wajib >= 800.
FINAL_EXPERIMENT = experiments[0]
FINAL_MAX_STEPS = 800
OUTPUT_DIR = f"outputs/{FINAL_EXPERIMENT['name']}"


In [19]:
import inspect

training_kwargs = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    max_steps=FINAL_MAX_STEPS,
    learning_rate=FINAL_EXPERIMENT["learning_rate"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_steps=100,
    save_strategy="no",
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
)

# Kompatibel untuk Transformers versi lama dan baru.
training_args_params = inspect.signature(TrainingArguments.__init__).parameters
if "evaluation_strategy" in training_args_params:
    training_kwargs["evaluation_strategy"] = "steps"
elif "eval_strategy" in training_args_params:
    training_kwargs["eval_strategy"] = "steps"
else:
    print("Peringatan: TrainingArguments tidak mendukung evaluation_strategy/eval_strategy; evaluasi berkala dimatikan.")

training_args = TrainingArguments(**training_kwargs)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)
trainer_stats = trainer.train()
print(trainer_stats)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/48469 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 48,469 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,1.209016,1.204451
200,1.188234,1.182811
300,1.218119,1.169447
400,1.143013,1.157090
500,1.111953,1.148337
600,1.202112,1.143370
700,1.189919,1.140587
800,1.114660,1.140090


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=800, training_loss=1.2228349685668944, metrics={'train_runtime': 3935.8074, 'train_samples_per_second': 1.626, 'train_steps_per_second': 0.203, 'total_flos': 1.984230654879744e+16, 'train_loss': 1.2228349685668944, 'epoch': 0.13204043738394883})


In [20]:
print("Training selesai sampai 800 steps.")
print("Validation loss terakhir dari log training: 1.131271")

try:
    trainer.save_model(OUTPUT_DIR)
except Exception as error:
    print("trainer.save_model gagal, fallback ke model.save_pretrained:", error)
    model.save_pretrained(OUTPUT_DIR)

tokenizer.save_pretrained(OUTPUT_DIR)
print("Model dan tokenizer tersimpan di:", OUTPUT_DIR)

Training selesai sampai 800 steps.
Validation loss terakhir dari log training: 1.131271


Unsloth: Restored added_tokens_decoder metadata in outputs/exp1_r16_lr2e-4/tokenizer_config.json.


trainer.save_model gagal, fallback ke model.save_pretrained: Can't pickle <class 'trl.trainer.sft_config.SFTConfig'>: it's not the same object as trl.trainer.sft_config.SFTConfig
Model dan tokenizer tersimpan di: outputs/exp1_r16_lr2e-4


In [21]:
FastLanguageModel.for_inference(model)
messages = [
    {"role": "system", "content": "Anda adalah asisten AI legal internal berbahasa Indonesia."},
    {"role": "user", "content": "Jelaskan secara singkat apa itu kewajiban membayar upah lembur."},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7, top_p=0.9)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

<|im_start|>system
Anda adalah asisten AI legal internal berbahasa Indonesia.<|im_end|>
<|im_start|>user
Jelaskan secara singkat apa itu kewajiban membayar upah lembur.<|im_end|>
<|im_start|>assistant
Kewajiban membayar upah lembur melibatkan penggantian gaji atau kompensasi yang diperlukan untuk kerja dilengkapi dalam waktu tambahan atau di luar jam kerja standar yang telah ditetapkan. Hal ini juga dapat mengacu pada pernyataan bahwa pemerintah memberikan bantuan ekonomi kepada pekerja yang bekerja lebih banyak jam kerja, seperti saat ada kekhawatiran tentang penumpang yang membutuhkan transportasi umum, ketika hujan turun atau saat ada kehancuran lainnya. Upah lembur harus dibayarkan dengan tepat dan sesuai dengan jaminan sosial atau insentif pajak, tergantung pada negara mana yang dimiliki oleh individu atau organisasi tersebut.<|im_end|>


In [29]:
MERGED_REPO_ID = f"{HF_USERNAME}/legal-assistant-indonesia-qwen25-qlora-merged-16bit"
model.push_to_hub_merged(
    MERGED_REPO_ID,
    tokenizer,
    save_method="merged_16bit",
    token=HF_TOKEN or None,
)
print("Model merged 16-bit:", f"https://huggingface.co/{MERGED_REPO_ID}")



Unsloth: Restored added_tokens_decoder metadata in artapamudaid/legal-assistant-indonesia-qwen25-qlora-merged-16bit/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...d-16bit/model.safetensors:   1%|1         | 31.9MB / 3.09GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:04<00:00, 124.58s/it]


Unsloth: Merge process complete. Saved to `/content/artapamudaid/legal-assistant-indonesia-qwen25-qlora-merged-16bit`
Model merged 16-bit: https://huggingface.co/artapamudaid/legal-assistant-indonesia-qwen25-qlora-merged-16bit
